In [ ]:
# Cell: Interactive Polygon Point Selector
# Run this cell to click on the video frame and get polygon_points
# for use in Cell 11.
#
# Controls:
#   Left Click  → add a point
#   Right Click → undo last point
#   C           → clear all points
#   Enter       → confirm polygon and print coordinates
#   Q           → quit without saving

import cv2
import numpy as np

# ── Load first frame ──
cap = cv2.VideoCapture(VIDEO_PATH)
ret, first_frame = cap.read()
cap.release()

if not ret:
    raise RuntimeError("Cannot read first frame from video.")

orig_h, orig_w = first_frame.shape[:2]
print(f"Video resolution: {orig_w} x {orig_h}")

# ── Scale to fit screen ──
SCREEN_W = 1366
scale     = SCREEN_W / orig_w
screen_h  = int(orig_h * scale)
display_frame = cv2.resize(first_frame, (SCREEN_W, screen_h))

print(f"Display size   : {SCREEN_W} x {screen_h}  (scale={scale:.4f})")
print("\nLeft Click = add point | Right Click = undo | C = clear | Enter = confirm | Q = quit\n")

polygon_display = []   # click coords in display space (for drawing)
polygon_orig    = []   # click coords mapped back to original resolution

WINDOW = "Polygon Selector  |  Enter=confirm  Q=quit"

def redraw():
    img = display_frame.copy()
    for idx, pt in enumerate(polygon_display):
        cv2.circle(img, pt, 6, (0, 255, 0), -1)
        cv2.circle(img, pt, 8, (255, 255, 255), 1)
        cv2.putText(img, str(idx + 1), (pt[0] + 10, pt[1] - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 255), 2)
        if idx > 0:
            cv2.line(img, polygon_display[idx - 1], pt, (0, 255, 0), 2)
    if len(polygon_display) > 2:
        cv2.line(img, polygon_display[-1], polygon_display[0], (0, 200, 0), 1)
    cv2.imshow(WINDOW, img)

def on_mouse(event, x, y, flags, param):
    ox = int(x / scale)
    oy = int(y / scale)
    if event == cv2.EVENT_LBUTTONDOWN:
        polygon_display.append((x, y))
        polygon_orig.append((ox, oy))
        print(f"  Point {len(polygon_orig):>2}: original=({ox}, {oy})")
        redraw()
    elif event == cv2.EVENT_RBUTTONDOWN and polygon_orig:
        removed = polygon_orig.pop()
        polygon_display.pop()
        print(f"  Removed: {removed}  |  {len(polygon_orig)} points remaining")
        redraw()

cv2.namedWindow(WINDOW, cv2.WINDOW_AUTOSIZE)
cv2.setMouseCallback(WINDOW, on_mouse)
cv2.imshow(WINDOW, display_frame)

confirmed = False
while True:
    key = cv2.waitKey(20) & 0xFF

    if key == 13:  # Enter — confirm
        if len(polygon_orig) >= 3:
            # Draw filled confirmation overlay
            img = display_frame.copy()
            pts = np.array(polygon_display, np.int32)
            overlay = img.copy()
            cv2.fillPoly(overlay, [pts], (0, 255, 0))
            cv2.addWeighted(overlay, 0.25, img, 0.75, 0, img)
            cv2.polylines(img, [pts], True, (0, 255, 0), 2)
            cv2.imshow(WINDOW, img)
            confirmed = True
            print("\nPolygon confirmed!")
            break
        else:
            print("Need at least 3 points.")

    elif key in (ord('c'), ord('C')):
        polygon_display.clear()
        polygon_orig.clear()
        redraw()
        print("  Cleared.")

    elif key in (ord('q'), ord('Q')):
        print("Quit — no polygon saved.")
        break

cv2.destroyAllWindows()

# ── Print coordinates ready to paste into Cell 11 ──
if confirmed and polygon_orig:
    print("\n" + "=" * 55)
    print(f"  {len(polygon_orig)} points at original resolution ({orig_w}x{orig_h})")
    print("=" * 55)
    print(f"\npolygon_points = {polygon_orig}")
    print(f"\nCopy the line above and paste it into Cell 11.")
    print("=" * 55)